# Práctica 6: pasajeros del aeropuerto de Asturias

Cada celda de código copiada de la teoría empieza con un comentario que indica el programa del que procede y, si se ha adaptado, qué se ha cambiado. Las respuestas se escriben en las celdas de texto marcadas como **Respuesta**.

| Contenido | Programa de teoría |
|---|---|
| Banco de pruebas: problemas, `evaluar`, `perfil`, `ejemplo` | `bateria.py` (sesiones 1 y 2) |
| Galería | Sesión 1 · `01-Bateria.py` |
| Gráfico estacional | Sesión 1 · `02-Ciclos.py` |
| Periodograma | Sesión 1 · `03-Periodograma.py` |
| Gráfico de desfase | Sesión 1 · `05-Memoria.py` |
| Mapa de características | Sesión 1 · `06-Mapa.py` |
| Referencias | Sesión 1 · `07-Referencias.py` |
| ETS | Sesión 1 · `08-ETS.py` |
| SARIMA | Sesión 1 · `09-SARIMA.py` |
| Theta | Sesión 1 · `10-Theta.py` |
| Comparación de métodos | Sesión 1 · `13-Comparacion.py` |
| Árboles potenciados | Sesión 2 · `51-Arboles.py` |
| DLinear | Sesión 2 · `52-DLinear.py` |
| Chronos-2 | Sesión 2 · `56-Chronos.py` |

## Banco de pruebas

In [ ]:
# Copiado de bateria.py (sesiones 1 y 2): la batería de ocho problemas.
# Adaptado: los datos se leen de la carpeta «datos» del cuaderno (si falta un fichero, se descarga como en bateria.py)
import io
import json
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DATOS = Path("datos")
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8a8a8a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "grid.linewidth": 0.6, "axes.axisbelow": True})


def descargar(nombre, url):
    """Descarga un fichero la primera vez y lo guarda en datos; después lo lee del disco."""
    fichero = DATOS / nombre
    if not fichero.exists():
        DATOS.mkdir(parents=True, exist_ok=True)
        peticion = urllib.request.Request(url, headers={"User-Agent": "visualizacion-datos-temporales/1.0 (docencia)"})
        fichero.write_bytes(urllib.request.urlopen(peticion, timeout=120).read())
    return fichero


def co2():
    """CO₂ medio mensual en Mauna Loa (ppm), NOAA, desde 1958."""
    df = pd.read_csv(descargar("co2_mlo.csv", "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"), comment="#")
    s = pd.Series(df["average"].values, index=pd.to_datetime(dict(year=df.year, month=df.month, day=1)))
    return s.where(s > 0).interpolate().asfreq("MS")


def eurostat(pais):
    """Pernoctaciones mensuales en hoteles de un país (millones), Eurostat (tour_occ_nim), desde 1990."""
    url = ("https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tour_occ_nim?unit=NR&nace_r2=I551"
           "&c_resid=TOTAL&format=JSON&lang=EN")
    d = json.loads(descargar("eurostat_pernoctaciones.json", url).read_text())
    paises, meses = d["dimension"]["geo"]["category"]["index"], d["dimension"]["time"]["category"]["index"]
    fila = paises[pais] * len(meses)
    s = pd.Series({pd.Timestamp(m + "-01"): d["value"].get(str(fila + j)) for m, j in meses.items()}, dtype=float)
    return (s / 1e6).asfreq("MS")


def pernoctaciones():
    return eurostat("ES")["1999":]


def manchas():
    """Número medio mensual de manchas solares, SILSO (Observatorio Real de Bélgica), desde 1749."""
    df = pd.read_csv(descargar("manchas.csv", "https://www.sidc.be/SILSO/INFO/snmtotcsv.php"), sep=";", header=None)
    s = pd.Series(df[3].values, index=pd.to_datetime(dict(year=df[0], month=df[1], day=1)))
    return s.where(s >= 0).interpolate().asfreq("MS")


def ree(nombre, ruta, titulo, años):
    """Una serie diaria de la API de Red Eléctrica (GWh), descargada año a año."""
    fichero = DATOS / f"{nombre}.csv"
    if not fichero.exists():
        filas = []
        for año in años:
            url = (f"https://apidatos.ree.es/es/datos/{ruta}?time_trunc=day"
                   f"&start_date={año}-01-01T00:00&end_date={año}-12-31T23:59")
            for grupo in json.loads(urllib.request.urlopen(url, timeout=120).read())["included"]:
                if grupo["attributes"]["title"] == titulo:
                    filas += [(v["datetime"][:10], v["value"] / 1000) for v in grupo["attributes"]["values"]]
        DATOS.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(filas, columns=["fecha", "GWh"]).to_csv(fichero, index=False)
    return pd.read_csv(fichero, index_col="fecha", parse_dates=True)["GWh"].asfreq("D")


def demanda():
    """Demanda eléctrica diaria de España (GWh), 2014-2025."""
    return ree("demanda", "demanda/evolucion", "Demanda", range(2014, 2026))


def eolica():
    """Producción eólica diaria de España (GWh), 2015-2025."""
    return ree("eolica", "generacion/estructura-generacion", "Eólica", range(2015, 2026))


def gripe():
    """Visitas diarias al artículo «Gripe» de la Wikipedia en español, 2016-2025."""
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/es.wikipedia/all-access/user/"
           "Gripe/daily/20160101/20251231")
    items = json.loads(descargar("wikipedia_gripe.json", url).read_text())["items"]
    return pd.Series({pd.Timestamp(i["timestamp"][:8]): i["views"] for i in items}, dtype=float).asfreq("D")


def meteo(nombre, lat, lon, variable, desde="1990-01-01"):
    """Una variable diaria del reanálisis ERA5 (Open-Meteo) en un punto, hasta 2025."""
    url = (f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
           f"&start_date={desde}&end_date=2025-12-31&daily={variable}&timezone=Europe%2FMadrid")
    d = json.loads(descargar(f"{nombre}.json", url).read_text())["daily"]
    return pd.Series(d[variable], index=pd.to_datetime(d["time"]), dtype=float).asfreq("D")


def lluvia():
    """Lluvia diaria en Oviedo (mm), 1990-2025."""
    return meteo("lluvia_oviedo", 43.36, -5.85, "precipitation_sum")


def euro_dolar():
    """Dólares por euro, cambio de referencia diario del BCE (días hábiles), 1999-2025."""
    z = zipfile.ZipFile(descargar("eurofxref-hist.zip", "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"))
    df = pd.read_csv(io.BytesIO(z.read(z.namelist()[0])), usecols=["Date", "USD"], parse_dates=["Date"])
    return df.set_index("Date")["USD"].sort_index()[:"2025-12-31"].dropna()


# H: horizonte; ciclo: periodo de la referencia estacional (None si no hay); periodo: el que usan los modelos
# estadísticos; ventana: valores pasados que reciben los modelos aprendidos; n y paso: número de orígenes de la
# evaluación y separación entre ellos
PROBLEMAS = {
    "CO₂ en Mauna Loa": dict(cargar=co2, unidad="ppm", H=12, ciclo=12, periodo=12, ventana=36, n=13, paso=3),
    "Pernoctaciones hoteleras": dict(cargar=pernoctaciones, unidad="millones", H=12, ciclo=12, periodo=12, ventana=36, n=13, paso=3),
    "Manchas solares": dict(cargar=manchas, unidad="número de manchas", H=24, ciclo=132, periodo=None, ventana=144, n=17, paso=6),
    "Demanda eléctrica": dict(cargar=demanda, unidad="GWh", H=14, ciclo=7, periodo=7, ventana=28, n=26, paso=28),
    "Producción eólica": dict(cargar=eolica, unidad="GWh", H=14, ciclo=None, periodo=None, ventana=28, n=26, paso=28),
    "Visitas a «Gripe»": dict(cargar=gripe, unidad="visitas", H=14, ciclo=7, periodo=7, ventana=28, n=26, paso=28),
    "Lluvia en Oviedo": dict(cargar=lluvia, unidad="mm", H=14, ciclo=None, periodo=None, ventana=28, n=26, paso=28),
    "Euro/dólar": dict(cargar=euro_dolar, unidad="USD por EUR", H=20, ciclo=None, periodo=None, ventana=40, n=26, paso=20),
}
BATERIA = list(PROBLEMAS)

In [ ]:
# Copiado de bateria.py (sesiones 1 y 2): el banco de pruebas y sus dos figuras.
# Adaptado: evaluar() no guarda los resultados en disco; perfil() y ejemplo() dibujan solo los problemas indicados
import time


def origenes(serie, p):
    """Posiciones de los orígenes de la evaluación: los n últimos, separados p["paso"] observaciones."""
    ultimo = len(serie) - p["H"]
    return [ultimo - k * p["paso"] for k in range(p["n"])][::-1]


def referencias(historia, p, H):
    """Tres predicciones de referencia: persistencia, estacional ingenua y climatología."""
    r = {"persistencia": np.repeat(historia.iloc[-1], H)}
    if p["ciclo"]:
        ultimo_ciclo = historia.iloc[-p["ciclo"]:].values
        r["estacional ingenua"] = np.resize(ultimo_ciclo, H)
    paso = historia.index[1] - historia.index[0] if isinstance(historia.index, pd.DatetimeIndex) else None
    fechas = pd.date_range(historia.index[-1] + paso, periods=H, freq=paso) if paso else None
    if fechas is not None and paso >= pd.Timedelta(days=28):   # mensual: media de ese mes
        medias = historia.groupby(historia.index.month).mean()
        r["climatología"] = medias.loc[fechas.month].values
    elif fechas is not None:                                   # diaria: media de ese día del año (±15 días)
        dia = historia.groupby(historia.index.dayofyear).mean()
        dia = pd.concat([dia.iloc[-15:], dia, dia.iloc[:15]]).rolling(31, center=True).mean().iloc[15:-15]
        r["climatología"] = dia.reindex(fechas.dayofyear).values
    return r


def evaluar(metodo, prever, ajustar=None, problemas=None):
    """Evalúa un método en la batería con origen móvil.
    prever(historia, H, p, modelo) devuelve un diccionario con "media" y, si el método da intervalos, "p10" y "p90".
    ajustar(historia, p), si se da, se llama una vez por problema con los datos anteriores al primer origen.
    Devuelve una tabla con el error escalado (MASE) y la cobertura del intervalo del 80 % por problema y horizonte,
    con las referencias incluidas, y un ejemplo de predicción por problema (el último origen)."""
    filas, ejemplos = [], {}
    for nombre in problemas or PROBLEMAS:
        p = PROBLEMAS[nombre]
        serie = p["cargar"]().dropna()
        orig = origenes(serie, p)
        # Escala del MASE: error medio de la persistencia a un paso en los datos anteriores al primer origen
        escala = serie.iloc[:orig[0]].diff().abs().mean()
        t0 = time.perf_counter()
        modelo = ajustar(serie.iloc[:orig[0]], p) if ajustar else None
        errores, dentro, err_ref = [], [], {}
        for o in orig:
            historia, real = serie.iloc[:o], serie.iloc[o:o + p["H"]].values
            pred = prever(historia, p["H"], p, modelo)
            errores.append(np.abs(np.asarray(pred["media"]) - real) / escala)
            if pred.get("p10") is not None:
                dentro.append((real >= pred["p10"]) & (real <= pred["p90"]))
            for ref, valor in referencias(historia, p, p["H"]).items():
                err_ref.setdefault(ref, []).append(np.abs(valor - real) / escala)
        segundos = time.perf_counter() - t0
        ejemplos[nombre] = (serie.iloc[max(0, o - 4 * p["H"]):o], serie.iloc[o:o + p["H"]], pred)
        cobertura = np.mean(dentro, axis=0) if dentro else np.full(p["H"], np.nan)
        for h in range(p["H"]):
            filas.append((metodo, nombre, h + 1, np.mean(errores, axis=0)[h], cobertura[h], segundos))
            filas += [(ref, nombre, h + 1, np.mean(e, axis=0)[h], np.nan, 0.0) for ref, e in err_ref.items()]
        texto = "sin intervalo" if np.isnan(cobertura).all() else f"{np.nanmean(cobertura):.0%}"
        print(f"{nombre:26s} MASE medio {np.mean(errores):5.2f}   persistencia {np.mean(err_ref['persistencia']):5.2f}"
              f"   cobertura del 80 %: {texto}   ({segundos:.1f} s)")
    tabla = pd.DataFrame(filas, columns=["metodo", "problema", "h", "mase", "cobertura", "segundos"])
    return tabla, ejemplos


def perfil(tabla, metodo, problemas):
    """Un panel por problema: error escalado por horizonte del método (color) y de las referencias (gris)."""
    fig, ejes = plt.subplots(1, len(problemas), figsize=(7 * len(problemas), 4), squeeze=False)
    for ax, nombre in zip(ejes.flat, problemas):
        t = tabla[tabla.problema == nombre]
        for ref, estilo in [("persistencia", "-"), ("estacional ingenua", "--"), ("climatología", ":")]:
            r = t[t.metodo == ref]
            if len(r):
                ax.plot(r.h, r.mase, estilo, color=GRIS, lw=1.3, label=ref)
        m = t[t.metodo == metodo]
        ax.plot(m.h, m.mase, "o-", color=NARANJA, ms=3, lw=2, label=metodo)
        ax.set_title(nombre, fontsize=10)
        # Escala vertical fijada por el método y la persistencia: una referencia mucho peor queda fuera del panel
        ax.set_ylim(0, 1.3 * max(m.mase.max(), t[t.metodo == "persistencia"].mase.max()))
        ax.set_xlabel("horizonte")
        ax.set_ylabel("error escalado (MASE)")
    ejes.flat[0].legend(fontsize=8)
    fig.suptitle(f"{metodo}: error por horizonte (gris: referencias)")
    plt.tight_layout()
    plt.show()


def ejemplo(ejemplos, nombres, metodo):
    """La predicción del último origen en algunos problemas, con su intervalo del 80 % si lo hay."""
    fig, ejes = plt.subplots(1, len(nombres), figsize=(max(9, 5 * len(nombres)), 4))
    for ax, nombre in zip(np.atleast_1d(ejes), nombres):
        historia, real, pred = ejemplos[nombre]
        fechas = real.index
        ax.plot(historia, color="black", lw=1)
        ax.plot(real, color="black", lw=1, ls=":")
        if pred.get("p10") is not None:
            ax.fill_between(fechas, pred["p10"], pred["p90"], color=NARANJA, alpha=0.25, lw=0)
        ax.plot(fechas, pred["media"], color=NARANJA, lw=2)
        ax.set_title(nombre, fontsize=10)
        ax.set_ylabel(PROBLEMAS[nombre]["unidad"])
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle(f"{metodo}: predicción desde el último origen (puntos: lo que ocurrió)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Problema de la práctica: pasajeros mensuales del aeropuerto de Asturias, con la misma forma que los de bateria.py
def aeropuerto():
    """Pasajeros mensuales del aeropuerto de Asturias (miles), Eurostat (avia_paoa), desde 2000."""
    url = ("https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/avia_paoa?freq=M&unit=PAS"
           "&tra_meas=PAS_CRD&rep_airp=ES_LEAS&schedule=TOTAL&tra_cov=TOTAL&format=JSON&lang=EN")
    d = json.loads(descargar("aeropuerto_asturias.json", url).read_text())
    meses = d["dimension"]["time"]["category"]["index"]
    s = pd.Series({pd.Timestamp(m + "-01"): d["value"].get(str(j)) for m, j in meses.items()}, dtype=float)
    return (s / 1000).dropna().asfreq("MS")


AEROPUERTO = "Pasajeros del aeropuerto de Asturias"
PROBLEMAS[AEROPUERTO] = dict(cargar=aeropuerto, unidad="miles de pasajeros", H=12, ciclo=12, periodo=12, ventana=36, n=13, paso=3)
pasajeros = aeropuerto()
print(f"{len(pasajeros)} meses, de {pasajeros.index[0]:%m/%Y} a {pasajeros.index[-1]:%m/%Y}")

## 1. Describir la serie

In [ ]:
# Copiado de Sesión 1 · 01-Bateria.py, adaptado a un problema: la serie completa (izquierda) y su tramo final,
# de la longitud de cuatro horizontes (derecha)
def galeria(nombre):
    p = PROBLEMAS[nombre]
    s = p["cargar"]().dropna()
    fig, (completa, final) = plt.subplots(1, 2, figsize=(15, 3.8), gridspec_kw={"width_ratios": [3, 1]})
    completa.plot(s, color=AZUL, lw=0.8)
    completa.set_title(f"{nombre}: {len(s)} valores, {s.index[0]:%Y}-{s.index[-1]:%Y}", fontsize=10, loc="left")
    completa.set_ylabel(p["unidad"], fontsize=9)
    final.plot(s.iloc[-4 * p["H"]:], color="black", lw=0.9)
    final.set_title(f"últimos {4 * p['H']} valores", fontsize=9)
    final.set_xticks([])
    plt.tight_layout()
    plt.show()


galeria(AEROPUERTO)

¿Qué ocurrió en 2020 y 2021? ¿Ha recuperado la serie su nivel anterior?

**Respuesta.**

In [ ]:
# Copiado de Sesión 1 · 02-Ciclos.py: gráfico estacional. Una línea por año sobre la posición dentro del año, con
# un color por año del más antiguo (morado) al más reciente (amarillo)
colores = lambda n: plt.cm.viridis(np.linspace(0, 0.95, n))


def por_año(ax, s, x, titulo, suavizado=1):
    """Una línea por año sobre la posición dentro del año (x), con un color por año del más antiguo (morado) al más reciente (amarillo)."""
    s = s.rolling(suavizado, center=True).mean() if suavizado > 1 else s
    años = sorted(set(s.index.year))
    for color, a in zip(colores(len(años)), años):
        t = s[str(a)]
        ax.plot(x(t.index), t.values, color=color, lw=0.8)
    ax.set_title(titulo, fontsize=11)


fig, ax = plt.subplots(figsize=(9, 5))
por_año(ax, pasajeros, lambda i: i.month, "Pasajeros del aeropuerto de Asturias")
ax.set_xticks(range(1, 13), list("EFMAMJJASOND"))
ax.set_ylabel("miles de pasajeros")
fig.colorbar(plt.cm.ScalarMappable(cmap="viridis"), ax=ax, shrink=0.8, pad=0.01,
             ticks=[0, 1]).set_ticklabels(["primer año", "último año"])
plt.show()

¿Qué se repite cada año? ¿Cambian la forma o la amplitud del ciclo con los años?

**Respuesta.**

In [ ]:
# Copiado de Sesión 1 · 03-Periodograma.py, adaptado a un problema. Periodograma de Welch de la serie sin su
# tendencia lineal, en escala doble logarítmica
from scipy.signal import welch, detrend


def periodograma(s, titulo, unidad):
    x = detrend(s.dropna().values)   # se quita la tendencia lineal, que dominaría los periodos largos
    f, potencia = welch(x, fs=1, nperseg=min(len(x), 2048 if len(x) > 4000 else len(x) // 2))
    periodo, potencia = 1 / f[1:], potencia[1:]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.loglog(periodo, potencia, color=AZUL, lw=1)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel(f"periodo ({unidad})")
    ax.set_ylabel("potencia (escala log)")
    plt.show()
    pico = periodo[np.argmax(potencia * (periodo < len(x) / 3))]
    print(f"periodo con más potencia: {pico:8.1f} {unidad}")


periodograma(pasajeros, AEROPUERTO, "meses")

¿En qué periodos tiene picos el periodograma? ¿Son estrechos o anchos?

**Respuesta.**

In [ ]:
# Copiado de Sesión 1 · 05-Memoria.py (gráficos de desfase), adaptado a un problema: cada punto es un par
# (valor anterior, valor) y la diagonal es la persistencia. Con miles de puntos se dibuja la densidad con hexágonos
def desfase(s, titulo):
    s = s.dropna()
    x, y = s.values[:-1], s.values[1:]
    fig, ax = plt.subplots(figsize=(6, 6))
    if len(s) < 1000:
        ax.scatter(x, y, s=10, color=AZUL, alpha=0.6)
    else:
        ax.hexbin(x, y, gridsize=35, bins="log", cmap="Blues", mincnt=1)
    lim = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lim, lim, color="black", lw=0.6, ls="--")
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("valor anterior")
    ax.set_ylabel("valor")
    plt.show()


desfase(pasajeros, AEROPUERTO)

¿Se parece cada mes al anterior? ¿Hay un único régimen o varios?

**Respuesta.**

## 2. Elegir los métodos

In [ ]:
# Copiado de Sesión 1 · 06-Mapa.py. Adaptado: las características se calculan con una función para poder añadir
# los problemas de la práctica, que se marcan en rojo en el mapa
from scipy.signal import periodogram
from statsmodels.tsa.seasonal import MSTL


def entropia_espectral(x):
    """Entropía del periodograma normalizado, entre 0 y 1. Baja: la varianza se concentra en pocos ciclos y la
    serie es previsible. Alta (1 = ruido blanco): la varianza se reparte por igual entre todos los periodos"""
    _, p = periodogram((x - x.mean()) / x.std())
    p = p[1:] / p[1:].sum()
    return -(p * np.log(p + 1e-12)).sum() / np.log(len(p))


# Ciclos de cada problema, los mismos que en 04-Descomposicion.py; la fuerza del ciclo es la del más fuerte
CICLOS = {"CO₂ en Mauna Loa": (12,), "Pernoctaciones hoteleras": (12,), "Manchas solares": (132,),
          "Demanda eléctrica": (7, 365), "Producción eólica": (7, 365), "Visitas a «Gripe»": (7, 365),
          "Lluvia en Oviedo": (7, 365), "Euro/dólar": (5, 260), AEROPUERTO: (12,)}


def caracteristicas(nombre):
    """Seis características de un problema. La tendencia y el ciclo se miden con MSTL en los 3000 últimos valores;
    las demás, sobre los cambios de un paso al siguiente."""
    s = PROBLEMAS[nombre]["cargar"]().dropna()
    reciente = s.iloc[-3000:]
    d = MSTL(np.log(reciente) if nombre == "Visitas a «Gripe»" else reciente, periods=CICLOS[nombre]).fit()
    estacional = d.seasonal.to_frame() if d.seasonal.ndim == 1 else d.seasonal
    f = lambda c: max(0, 1 - np.var(d.resid) / np.var(c + d.resid))
    cambios = s.diff().dropna()
    return {"fuerza de la tendencia": f(d.trend),
            "fuerza del ciclo": max(f(estacional.iloc[:, k]) for k in range(estacional.shape[1])),
            "entropía espectral": entropia_espectral(cambios.values),
            "autocorrelación de los cambios": cambios.autocorr(1),
            "curtosis de los cambios": cambios.kurt(), "proporción de ceros": (s == 0).mean()}


def mapa(tabla, destacados):
    """Mapa de la batería: fuerza del ciclo frente a entropía espectral de los cambios. Área del círculo: fuerza de
    la tendencia; color: curtosis de los cambios. En rojo, los problemas destacados."""
    fig, ax = plt.subplots(figsize=(10, 6.5))
    area = 200 + 1200 * tabla["fuerza de la tendencia"]
    ax.scatter(tabla["fuerza del ciclo"], tabla["entropía espectral"], s=area, c=tabla["curtosis de los cambios"].clip(0, 30),
               cmap="Oranges", vmin=0, vmax=30, edgecolor=["red" if n in destacados else "black" for n in tabla.index],
               linewidth=[2 if n in destacados else 1 for n in tabla.index])
    desplazamiento = {"Euro/dólar": (-20, 18), "Lluvia en Oviedo": (-70, -26), "Producción eólica": (10, 12),
                      "Visitas a «Gripe»": (16, -18), "Demanda eléctrica": (-40, -28), "CO₂ en Mauna Loa": (15, 5),
                      "Precio de la electricidad": (-60, -36)}
    for nombre, fila in tabla.iterrows():
        ax.annotate(nombre, (fila["fuerza del ciclo"], fila["entropía espectral"]), xytext=desplazamiento.get(nombre, (14, 8)),
                    textcoords="offset points", fontsize=10, color="red" if nombre in destacados else "black")
    ax.set_xlabel("fuerza del ciclo (0: sin ciclo; 1: el ciclo domina)")
    ax.set_ylabel("entropía espectral de los cambios (0: un solo ciclo; 1: ruido)")
    ax.set_xlim(-0.05, 1.3)
    ax.set_ylim(0.2, 1.05)
    fig.colorbar(plt.cm.ScalarMappable(cmap="Oranges", norm=plt.Normalize(0, 30)), ax=ax, label="curtosis de los cambios (saltos bruscos)")
    ax.set_title("Mapa de la batería: área del círculo = fuerza de la tendencia")
    plt.tight_layout()
    plt.show()


tabla_caracteristicas = pd.DataFrame({nombre: caracteristicas(nombre) for nombre in BATERIA + [AEROPUERTO]}).T
print(tabla_caracteristicas.round(2).to_string())
mapa(tabla_caracteristicas, [AEROPUERTO])

¿A qué problemas de la batería se parece la serie?

**Respuesta.**

### Métodos disponibles

En estas celdas solo se definen las funciones de cada método; se evalúan en el apartado 3.

In [ ]:
# Copiado de Sesión 1 · 08-ETS.py (funciones ajustar y prever, renombradas ajustar_ets y prever_ets)
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

warnings.filterwarnings("ignore")


# ETS: suavizado exponencial en forma de modelo estadístico (error, tendencia y ciclo). Cada componente puede ser
# aditiva, multiplicativa o no existir, y la tendencia, amortiguada. La configuración se elige por AIC una vez por
# problema, con los datos anteriores al primer origen; en cada origen se reajusta con los últimos 1500 valores
def ajustar_ets(historia, p):
    y = historia.iloc[-1500:]
    ciclos = [None] + (["add", "mul"] if p["periodo"] else [])
    candidatos = [dict(trend=t, damped_trend=a, seasonal=c, seasonal_periods=p["periodo"])
                  for t, a in [(None, False), ("add", False), ("add", True)] for c in ciclos
                  if not (c == "mul" and y.min() <= 0)]            # el ciclo multiplicativo no admite ceros
    aic = [ETSModel(y.values, error="add", **c).fit(disp=False).aic for c in candidatos]
    elegido = candidatos[int(np.argmin(aic))]
    print(f"   configuración elegida: tendencia {elegido['trend']}, amortiguada {elegido['damped_trend']}, ciclo {elegido['seasonal']}")
    return elegido


def prever_ets(historia, H, p, config):
    y = pd.Series(historia.iloc[-1500:].values)   # índice entero: las fechas no hacen falta
    ajuste = ETSModel(y, error="add", **config).fit(disp=False)
    marco = ajuste.get_prediction(start=len(y), end=len(y) + H - 1).summary_frame(alpha=0.2)
    return {"media": marco["mean"].values, "p10": marco["pi_lower"].values, "p90": marco["pi_upper"].values}

In [ ]:
# Copiado de Sesión 1 · 09-SARIMA.py (funciones ajustar y prever, renombradas ajustar_sarima y prever_sarima)
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import kpss

warnings.filterwarnings("ignore")


# SARIMA(p,d,q)(P,D,Q,s). La diferencia d se decide con la prueba KPSS (d = 1 si la serie no es estacionaria);
# D = 1 si el problema tiene ciclo. Con d y D fijos, los órdenes se eligen por AIC entre unos pocos candidatos.
# Los parámetros se estiman una vez, con los datos anteriores al primer origen; en cada origen solo se actualiza
# el estado del modelo con los datos nuevos (filter), que es mucho más rápido que reajustarlo
def ajustar_sarima(historia, p):
    y = pd.Series(historia.iloc[-1500:].values)
    d = int(kpss(y, nlags="auto")[1] < 0.05)
    s = p["periodo"]
    estacionales = [(P, 1, Q, s) for P, Q in [(0, 1), (1, 1)]] if s else [(0, 0, 0, 0)]
    mejor = None
    for (a, m), est in itertools.product([(1, 0), (0, 1), (1, 1), (2, 1)], estacionales):
        ajuste = SARIMAX(y, order=(a, d, m), seasonal_order=est).fit(disp=False)
        if mejor is None or ajuste.aic < mejor[0]:
            mejor = (ajuste.aic, (a, d, m), est, ajuste.params)
    print(f"   SARIMA{mejor[1]}{mejor[2] if s else ''}")
    return mejor[1:]


def prever_sarima(historia, H, p, modelo):
    orden, estacional, parametros = modelo
    y = pd.Series(historia.iloc[-1500:].values)
    pred = SARIMAX(y, order=orden, seasonal_order=estacional).filter(parametros).get_forecast(H)
    intervalo = pred.conf_int(alpha=0.2).values
    return {"media": pred.predicted_mean.values, "p10": intervalo[:, 0], "p90": intervalo[:, 1]}

In [ ]:
# Copiado de Sesión 1 · 10-Theta.py (función prever, renombrada prever_theta)
from statsmodels.tsa.forecasting.theta import ThetaModel

warnings.filterwarnings("ignore")


# Método Theta (Assimakopoulos y Nikolopoulos, 2000), el ganador de la competición M3. Quita el ciclo, combina
# una recta de tendencia con un suavizado exponencial simple y vuelve a poner el ciclo. No hay nada que elegir:
# es la referencia «sencilla pero difícil de batir» de las competiciones de predicción
def prever_theta(historia, H, p, modelo):
    y = pd.Series(historia.iloc[-1500:].values)
    con_ciclo = bool(p["periodo"])
    metodo = "multiplicative" if con_ciclo and y.min() > 0 else "additive"   # el multiplicativo no admite ceros
    ajuste = ThetaModel(y, period=p["periodo"] or 1, deseasonalize=con_ciclo, method=metodo).fit()
    intervalo = ajuste.prediction_intervals(H, alpha=0.2)
    return {"media": ajuste.forecast(H).values, "p10": intervalo["lower"].values, "p90": intervalo["upper"].values}

In [ ]:
# Copiado de Sesión 2 · 51-Arboles.py (funciones ajustar y prever, renombradas ajustar_arboles y prever_arboles)
from sklearn.ensemble import HistGradientBoostingRegressor


# Árboles de decisión potenciados con un único modelo para todos los horizontes. Cada fila de entrenamiento es un
# par (origen t, horizonte h): las variables son los últimos «ventana» valores hasta t, el horizonte h y el mes y
# el día de la semana de la fecha que se predice; el objetivo es el cambio y(t+h) − y(t). Predecir el cambio y no
# el valor permite al árbol, que no extrapola, seguir a una serie con tendencia
def fila(y, fechas, t, h, ventana):
    pasado = y[t - ventana + 1:t + 1] - y[t]
    calendario = [fechas[t + h].month, fechas[t + h].dayofweek] if fechas is not None else [0, 0]
    return np.concatenate([pasado, [y[t], h], calendario])


def fechas_con_futuro(historia, H):
    """Fechas de la historia y de los H pasos siguientes (None si la serie no tiene frecuencia fija)."""
    if historia.index.freq is None:
        return None
    return historia.index.append(pd.date_range(historia.index[-1], periods=H + 1, freq=historia.index.freq)[1:])


def ajustar_arboles(historia, p):
    historia = historia.iloc[-3000:]
    y, fechas, v, H = historia.values, fechas_con_futuro(historia, 0), p["ventana"], p["H"]
    pares = [(t, h) for t in range(v - 1, len(y) - H) for h in range(1, H + 1)]
    X = np.array([fila(y, fechas, t, h, v) for t, h in pares])
    Y = np.array([y[t + h] - y[t] for t, h in pares])
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(X, Y)


def prever_arboles(historia, H, p, arboles):
    y, fechas, t = historia.values, fechas_con_futuro(historia, H), len(historia) - 1
    X = np.array([fila(y, fechas, t, h, p["ventana"]) for h in range(1, H + 1)])
    return {"media": y[t] + arboles.predict(X)}

In [ ]:
# Copiado de Sesión 2 · 52-DLinear.py (funciones ajustar y prever, renombradas ajustar_dlinear y prever_dlinear)
import torch


# DLinear (Zeng y otros, 2023): la ventana de entrada se separa en tendencia (media móvil) y resto, y cada parte
# pasa por una capa lineal que da los H valores futuros. Es un modelo autorregresivo lineal con una descomposición
# previa; su artículo mostró que igualaba a los transformers diseñados para series en sus propios conjuntos de prueba
class DLinear(torch.nn.Module):
    def __init__(self, ventana, H, k=25):
        super().__init__()
        self.k = min(k, ventana) // 2 * 2 + 1                     # longitud impar de la media móvil
        self.tendencia, self.resto = torch.nn.Linear(ventana, H), torch.nn.Linear(ventana, H)

    def forward(self, x):                                          # x: (ejemplos, ventana)
        m = self.k // 2
        relleno = torch.cat([x[:, :1].repeat(1, m), x, x[:, -1:].repeat(1, m)], dim=1)
        media = relleno.unfold(1, self.k, 1).mean(dim=2)
        return self.tendencia(media) + self.resto(x - media)


def ajustar_dlinear(historia, p):
    y = historia.values[-3000:]
    media, desv = y.mean(), y.std()
    z = torch.tensor((y - media) / desv, dtype=torch.float32)
    v, H = p["ventana"], p["H"]
    X = torch.stack([z[t - v:t] for t in range(v, len(z) - H + 1)])
    Y = torch.stack([z[t:t + H] for t in range(v, len(z) - H + 1)])
    torch.manual_seed(0)
    modelo = DLinear(v, H)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    for epoca in range(300):                                       # todos los ejemplos en cada paso
        error = ((modelo(X) - Y) ** 2).mean()
        optimizador.zero_grad()
        error.backward()
        optimizador.step()
    return modelo, media, desv


def prever_dlinear(historia, H, p, ajustado):
    modelo, media, desv = ajustado
    x = torch.tensor((historia.values[-p["ventana"]:] - media) / desv, dtype=torch.float32)[None]
    with torch.no_grad():
        return {"media": modelo(x)[0].numpy() * desv + media}

In [ ]:
# Métodos disponibles: nombre -> (prever, ajustar)
MENU = {"ETS": (prever_ets, ajustar_ets),
        "SARIMA": (prever_sarima, ajustar_sarima),
        "Theta": (prever_theta, None),
        "árboles potenciados": (prever_arboles, ajustar_arboles),
        "DLinear": (prever_dlinear, ajustar_dlinear)}

Elegir dos o tres métodos de `MENU` y justificar la elección con las características de la serie y la tabla «campo de aplicación» de las transparencias. La elección y su justificación se escriben antes de evaluar los métodos.

**Respuesta.**

In [ ]:
ELEGIDOS = []   # dos o tres nombres de MENU

## 3. Evaluar los métodos

In [ ]:
# Copiado de Sesión 1 · 07-Referencias.py, adaptado a un problema. evaluar() calcula siempre las tres referencias:
# persistencia (el último valor), estacional ingenua (el último ciclo) y climatología (la media de esa época del año)
persistencia = lambda historia, H, p, modelo: {"media": np.repeat(historia.iloc[-1], H)}
tabla_referencias, _ = evaluar("referencias", persistencia, problemas=[AEROPUERTO])
referencias_aeropuerto = tabla_referencias[tabla_referencias.metodo != "referencias"]
print(referencias_aeropuerto.groupby("metodo").mase.mean()[["persistencia", "estacional ingenua", "climatología"]].round(2).to_string())

In [ ]:
# Evaluación de los métodos elegidos con el banco de pruebas (bateria.py) y perfil de error de cada uno
tablas, ejemplos = {}, {}
for nombre in ELEGIDOS:
    prever, ajustar = MENU[nombre]
    tablas[nombre], ejemplos[nombre] = evaluar(nombre, prever, ajustar, problemas=[AEROPUERTO])
    perfil(tablas[nombre], nombre, [AEROPUERTO])

In [ ]:
# Copiado de Sesión 1 · 13-Comparacion.py, adaptado a un problema: MASE medio y cobertura del intervalo del 80 %
# de las referencias y de los métodos evaluados, y su error por horizonte
def comparar(nombre, tabla_referencias, tablas):
    todo = pd.concat([tabla_referencias] + list(tablas.values()))
    todo = todo[(todo.problema == nombre) & (todo.metodo != "referencias")].drop_duplicates(["metodo", "h"])
    metodos = ["persistencia", "estacional ingenua", "climatología"] + [m for m in tablas if m in set(todo.metodo)]
    metodos = [m for m in metodos if m in set(todo.metodo)]
    resumen = todo.groupby("metodo").agg(MASE=("mase", "mean"), cobertura=("cobertura", "mean")).loc[metodos]
    print(resumen.round(2).to_string())
    t = todo.pivot(index="h", columns="metodo", values="mase")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for ref, estilo in [("persistencia", "-"), ("estacional ingenua", "--"), ("climatología", ":")]:
        if ref in t:
            ax.plot(t.index, t[ref], estilo, color=GRIS, lw=1.5, label=ref)
    modelos = [m for m in metodos if m in tablas]
    for m, color in zip(modelos, ["#2a78d6", "#eb6834", "#9b59b6", "#2ca02c", "#d62728", "#8c564b"]):
        ax.plot(t.index, t[m], "o-", color=color, ms=3, lw=2, label=m)
    ax.set_ylim(0, 1.3 * t[modelos + ["persistencia"]].max().max())
    ax.set_xlabel("horizonte")
    ax.set_ylabel("error escalado (MASE)")
    ax.set_title(f"{nombre}: error por horizonte (gris: referencias)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    return resumen


resumen_aeropuerto = comparar(AEROPUERTO, tabla_referencias, tablas)

¿Cuál es la mejor referencia? ¿Por qué?

**Respuesta.**

¿Mejoran los métodos elegidos a la persistencia y a la mejor referencia? ¿En qué horizontes?

**Respuesta.**

¿Coincide el resultado con lo que se esperaba en el apartado 2? Si no coincide, ¿por qué?

**Respuesta.**

## 4. Predicción para el aeropuerto

In [ ]:
RECOMENDADO = None   # uno de los métodos evaluados

In [ ]:
# Predicción del método recomendado desde el último origen, con su intervalo del 80 % si lo da (ejemplo() de bateria.py)
if RECOMENDADO:
    ejemplo(ejemplos[RECOMENDADO], [AEROPUERTO], RECOMENDADO)
    print(f"{RECOMENDADO}: MASE medio {resumen_aeropuerto.loc[RECOMENDADO, 'MASE']:.2f}; "
          f"cobertura del intervalo del 80 % en la evaluación: {resumen_aeropuerto.loc[RECOMENDADO, 'cobertura']:.0%}")

Recomendación al aeropuerto: qué método usar y cuánto puede equivocarse.

**Respuesta.**

## Ampliación opcional

### 5. Chronos-2

In [ ]:
# Copiado de Sesión 2 · 56-Chronos.py (función prever, renombrada prever_chronos)
from chronos import BaseChronosPipeline

# Chronos-2 (Amazon, 2025): un modelo preentrenado con miles de series de todo tipo. No se entrena con los datos
# del problema: en cada origen recibe los últimos 2048 valores (el contexto) y devuelve cuantiles de los H
# siguientes
modelo_chronos = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")   # unos 500 MB la primera vez


def prever_chronos(historia, H, p, _):
    contexto = historia.iloc[-2048:]
    df = pd.DataFrame({"item_id": 0, "timestamp": pd.RangeIndex(len(contexto)), "target": contexto.values})
    df["timestamp"] = pd.date_range("2000-01-01", periods=len(contexto), freq="D")   # fechas ficticias: solo cuenta el orden
    salida = modelo_chronos.predict_df(df, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
    return {"media": salida["0.5"].values, "p10": salida["0.1"].values, "p90": salida["0.9"].values}


MENU["Chronos-2"] = (prever_chronos, None)

In [ ]:
# Evaluación de Chronos-2 en el aeropuerto con el banco de pruebas

¿Cómo se compara Chronos-2 con los métodos del apartado 3?

**Respuesta.**

### 6. Precio de la electricidad

Precio medio diario del mercado eléctrico peninsular de 2014 a 2025, con un horizonte de 14 días.

In [ ]:
# Segundo problema: precio medio diario del mercado eléctrico peninsular, con la misma forma que los de bateria.py
def precio():
    """Precio medio diario del mercado eléctrico peninsular (€/MWh): media de los precios horarios del mercado
    diario publicados por Red Eléctrica (apidatos.ree.es, mercados/precios-mercados-tiempo-real), 2014-2025."""
    return pd.read_csv(DATOS / "precio_diario.csv", index_col="fecha", parse_dates=True)["eur_mwh"].asfreq("D")


PRECIO = "Precio de la electricidad"
PROBLEMAS[PRECIO] = dict(cargar=precio, unidad="€/MWh", H=14, ciclo=7, periodo=7, ventana=28, n=26, paso=28)
precios = precio()
print(f"{len(precios)} días, del {precios.index[0]:%d/%m/%Y} al {precios.index[-1]:%d/%m/%Y}")
galeria(PRECIO)

In [ ]:
# Gráfico estacional (por_año, de Sesión 1 · 02-Ciclos.py): media de 7 días, sobre el día del año
fig, ax = plt.subplots(figsize=(9, 5))
por_año(ax, precios, lambda i: i.dayofyear, "Precio de la electricidad (media de 7 días)", 7)
ax.set_xticks([1, 91, 182, 274], ["ene", "abr", "jul", "oct"])
ax.set_ylabel("€/MWh")
fig.colorbar(plt.cm.ScalarMappable(cmap="viridis"), ax=ax, shrink=0.8, pad=0.01,
             ticks=[0, 1]).set_ticklabels(["primer año", "último año"])
plt.show()
periodograma(precios, PRECIO, "días")

In [ ]:
desfase(precios, PRECIO)


# Adaptado de Sesión 1 · 05-Memoria.py: el gráfico de desfase de 2023-2025 con un color según el día de la semana
# del valor
def desfase_por_dia(s, desde):
    s = s[desde:]
    x, y, dia = s.values[:-1], s.values[1:], s.index[1:].dayofweek
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for m, color, etiqueta in [((dia >= 1) & (dia <= 4), AZUL, "martes a viernes"), (dia == 5, NARANJA, "sábado"),
                               (dia == 6, "red", "domingo"), (dia == 0, "green", "lunes")]:
        ax.scatter(x[m], y[m], s=6, color=color, alpha=0.5, label=etiqueta)
    lim = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lim, lim, color="black", lw=0.6, ls="--")
    ax.set_xlabel("valor anterior")
    ax.set_ylabel("valor")
    ax.set_title(f"{PRECIO} desde {desde}: color según el día de la semana del valor", fontsize=10)
    ax.legend(fontsize=8, markerscale=3)
    plt.show()


desfase_por_dia(precios, "2023")

Preguntas del apartado 1 para el precio. En el gráfico de desfase con los pares de días separados por el día de la semana, ¿se puede modelar la serie con un único perfil o hacen falta varios?

**Respuesta.**

In [ ]:
# Mapa de la batería (06-Mapa.py) con los dos problemas de la práctica
CICLOS[PRECIO] = (7, 365)
tabla_caracteristicas.loc[PRECIO] = caracteristicas(PRECIO)
print(tabla_caracteristicas.loc[[AEROPUERTO, PRECIO]].round(2).to_string())
mapa(tabla_caracteristicas, [AEROPUERTO, PRECIO])

¿A qué problemas de la batería se parece? Elegir los métodos y justificar la elección antes de evaluarlos.

**Respuesta.**

In [ ]:
ELEGIDOS_PRECIO = []   # nombres de MENU

In [ ]:
# Referencias (07-Referencias.py) y métodos elegidos para el precio, con el banco de pruebas
tabla_referencias_precio, _ = evaluar("referencias", persistencia, problemas=[PRECIO])
tablas_precio, ejemplos_precio = {}, {}
for nombre in ELEGIDOS_PRECIO:
    prever, ajustar = MENU[nombre]
    tablas_precio[nombre], ejemplos_precio[nombre] = evaluar(nombre, prever, ajustar, problemas=[PRECIO])
    perfil(tablas_precio[nombre], nombre, [PRECIO])
resumen_precio = comparar(PRECIO, tabla_referencias_precio, tablas_precio)

Preguntas del apartado 3 para el precio. ¿Cuánto mejoran los métodos a la persistencia?

**Respuesta.**